<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/DS_PROJECT/ds_proj_meth/CRISP_DM_Project_Sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект: Анализ выживаемости пассажиров Титаника  
**Автор:** Кондратьев Степан  
**Дата:** 2025 март  
**Цель:** Использование методов машинного обучения для прогнозирования выживаемости пассажиров Титаника на основе их характеристик, таких как пол, возраст, класс кают и других факторов.  

## 1. Понимание бизнеса (Business Understanding)

**Цель проекта:** Прогнозирование выживаемости пассажиров Титаника на основе их характеристик.  

**Постановка задачи:** Бинарная классификация пассажиров на выживших (1) и погибших (0).  

**Критерии успеха:**  
- **Метрики:**  
  - Точность (Accuracy)  
  - F1-мера (F1-Score)  
  - ROC-AUC  
- **Бизнес-результат:**  
  - Понимание ключевых факторов, влияющих на выживаемость, для улучшения безопасности пассажиров в будущем.  
  - Создание модели, способной предсказывать вероятность выживания с высокой точностью.  

## 2. Понимание данных (Data Understanding)

### Импорт библиотек

In [1]:
# Импорт библиотек
import polars as pl  # Для работы с данными: чтение, обработка и анализ

### Загрудка данных

In [2]:
# Настройка отображения всех столбцов в Polars
pl.Config.set_tbl_cols(-1)  # Показывать все столбцы

# Загрузка данных
train_url = "https://raw.githubusercontent.com/stefkong1982/netology.ru/refs/heads/Master/DS_PROJECT/ds_proj_meth/train.csv"
test_url = "https://raw.githubusercontent.com/stefkong1982/netology.ru/refs/heads/Master/DS_PROJECT/ds_proj_meth/test.csv"

train_df = pl.read_csv(train_url)
test_df = pl.read_csv(test_url)

In [28]:
# Вывод данных для первичного анализа
print("Данные обучающей выборки:")
train_df

Данные обучающей выборки:


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,"""Unknown""","""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,"""Unknown""","""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,"""Unknown""","""S"""
…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",27.0,0,0,"""211536""",13.0,"""Unknown""","""S"""
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",19.0,0,0,"""112053""",30.0,"""B42""","""S"""
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",28.0,1,2,"""W./C. 6607""",23.45,"""Unknown""","""S"""


Данные обучающей выборки:
shape: (891, 12)

| PassengerId | Survived | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket       | Fare   | Cabin  | Embarked |
|-------------|----------|--------|------------------------------------------------|--------|-------|-------|-------|--------------|--------|--------|----------|
| i64         | i64      | i64    | str                                            | str    | f64   | i64   | i64   | str          | f64    | str    | str      |
| 1           | 0        | 3      | "Braund, Mr. Owen Harris"                     | "male" | 22.0  | 1     | 0     | "A/5 21171"  | 7.25   | "Unknown" | "S"      |
| 2           | 1        | 1      | "Cumings, Mrs. John Bradley (Fl..."           | "female" | 38.0 | 1     | 0     | "PC 17599"   | 71.2833| "C85"  | "C"      |
| 3           | 1        | 3      | "Heikkinen, Miss. Laina"                      | "female" | 26.0 | 0     | 0     | "STON/O2. 3101282" | 7.925 | "Unknown" | "S"      |
| 4           | 1        | 1      | "Futrelle, Mrs. Jacques Heath..."             | "female" | 35.0 | 1     | 0     | "113803"     | 53.1   | "C123" | "S"      |
| 5           | 0        | 3      | "Allen, Mr. William Henry"                    | "male" | 35.0  | 0     | 0     | "373450"     | 8.05   | "Unknown" | "S"      |

Данные тестовой выборки:
shape: (418, 11)

| PassengerId | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket    | Fare    | Cabin | Embarked |
|-------------|--------|------------------------------------------------|--------|-------|-------|-------|-----------|---------|-------|----------|
| i64         | i64    | str                                            | str    | f64   | i64   | i64   | str       | f64     | str   | str      |
| 892         | 3      | "Kelly, Mr. James"                            | "male" | 34.5  | 0     | 0     | "330911"  | 7.8292  | null  | "Q"      |
| 893         | 3      | "Wilkes, Mrs. James (Ellen Needham)"          | "female" | 47.0 | 1     | 0     | "363272"  | 7.0     | null  | "S"      |
| 894         | 2      | "Myles, Mr. Thomas Francis"                   | "male" | 62.0  | 0     | 0     | "240276"  | 9.6875  | null  | "Q"      |
| 895         | 3      | "Wirz, Mr. Albert"                            | "male" | 27.0  | 0     | 0     | "315154"  | 8.6625  | null  | "S"      |
| 896         | 3      | "Hirvonen, Mrs. Alexander (Helga)"            | "female" | 22.0 | 1     | 1     | "3101298" | 12.2875 | null  | "S"      |

#### Данные обучающей выборки:

```
| PassengerId | Survived | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket      | Fare   | Cabin  | Embarked |  
|-------------|----------|--------|------------------------------------------------|--------|-------|-------|-------|-------------|--------|--------|----------|  
| i64         | i64      | i64    | str                                            | str    | f64   | i64   | i64   | str         | f64    | str    | str      |  
| 1           | 0        | 3      | "Braund, Mr. Owen Harris"                     | "male" | 22.0  | 1     | 0     | "A/5 21171" | 7.25   | "Unknown" | "S"     |  
| 2           | 1        | 1      | "Cumings, Mrs. John Bradley (Florence Briggs)" | "female" | 38.0 | 1     | 0     | "PC 17599"  | 71.2833 | "C85"  | "C"     |  
| 3           | 1        | 3      | "Heikkinen, Miss. Laina"                      | "female" | 26.0 | 0     | 0     | "STON/O2. 3101282" | 7.925 | "Unknown" | "S"     |  
| 4           | 1        | 1      | "Futrelle, Mrs. Jacques Heath (Lily May Peel)" | "female" | 35.0 | 1     | 0     | "113803"    | 53.1   | "C123" | "S"     |  
| 5           | 0        | 3      | "Allen, Mr. William Henry"                    | "male" | 35.0  | 0     | 0     | "373450"    | 8.05   | "Unknown" | "S"     |  
```

Данные обучающей выборки:  
Размер: (891, 12)  

| PassengerId | Survived | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket              | Fare     | Cabin   | Embarked |
|-------------|----------|--------|------------------------------------------------|--------|-------|-------|-------|---------------------|----------|---------|----------|
| i64         | i64      | i64    | str                                            | str    | f64   | i64   | i64   | str                 | f64      | str     | str      |
| 1           | 0        | 3      | "Braund, Mr. Owen Harris"                     | "male" | 22.0  | 1     | 0     | "A/5 21171"         | 7.25     | "Unknown" | "S"      |
| 2           | 1        | 1      | "Cumings, Mrs. John Bradley (Flo)             | "female" | 38.0  | 1     | 0     | "PC 17599"          | 71.2833  | "C85"   | "C"      |
| 3           | 1        | 3      | "Heikkinen, Miss. Laina"                      | "female" | 26.0  | 0     | 0     | "STON/O2. 3101282"  | 7.925    | "Unknown" | "S"      |
| 4           | 1        | 1      | "Futrelle, Mrs. Jacques Heath (Flo)           | "female" | 35.0  | 1     | 0     | "113803"            | 53.1     | "C123"  | "S"      |
| 5           | 0        | 3      | "Allen, Mr. William Henry"                    | "male" | 35.0  | 0     | 0     | "373450"            | 8.05     | "Unknown" | "S"      |


In [29]:
print("\nДанные тестовой выборки:")
test_df


Данные тестовой выборки:


PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,str,str,f64,i64,i64,str,f64,str,str
892,3,"""Kelly, Mr. James""","""male""",34.5,0,0,"""330911""",7.8292,null,"""Q"""
893,3,"""Wilkes, Mrs. James (Ellen Need…","""female""",47.0,1,0,"""363272""",7.0,null,"""S"""
894,2,"""Myles, Mr. Thomas Francis""","""male""",62.0,0,0,"""240276""",9.6875,null,"""Q"""
895,3,"""Wirz, Mr. Albert""","""male""",27.0,0,0,"""315154""",8.6625,null,"""S"""
896,3,"""Hirvonen, Mrs. Alexander (Helg…","""female""",22.0,1,1,"""3101298""",12.2875,null,"""S"""
…,…,…,…,…,…,…,…,…,…,…
1305,3,"""Spector, Mr. Woolf""","""male""",null,0,0,"""A.5. 3236""",8.05,null,"""S"""
1306,1,"""Oliva y Ocana, Dona. Fermina""","""female""",39.0,0,0,"""PC 17758""",108.9,"""C105""","""C"""
1307,3,"""Saether, Mr. Simon Sivertsen""","""male""",38.5,0,0,"""SOTON/O.Q. 3101262""",7.25,null,"""S"""


| PassengerId | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket   | Fare     | Cabin | Embarked |
|-------------|--------|------------------------------------------------|--------|-------|-------|-------|----------|----------|-------|----------|
| i64         | i64    | str                                            | str    | f64   | i64   | i64   | str      | f64      | str   | str      |
| 892         | 3      | "Kelly, Mr. James"                           | "male" | 34.5  | 0     | 0     | "330911" | 7.8292   | null  | "Q"      |
| 893         | 3      | "Wilkes, Mrs. James (Ellen Needham)"        | "female" | 47.0  | 1     | 0     | "363272" | 7.0      | null  | "S"      |
| 894         | 2      | "Myles, Mr. Thomas Francis"                   | "male" | 62.0  | 0     | 0     | "240276" | 9.6875   | null  | "Q"      |
| 895         | 3      | "Wirz, Mr. Albert"                           | "male" | 27.0  | 0     | 0     | "315154" | 8.6625   | null  | "S"      |
| 896         | 3      | "Hirvonen, Mrs. Alexander (Helga)"           | "female" | 22.0  | 1     | 1     | "3101298"| 12.2875  | null  | "S"      |

Данные тестовой выборки:  
Размер: (418, 11)  

| PassengerId | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket   | Fare     | Cabin | Embarked |
|-------------|--------|------------------------------------------------|--------|-------|-------|-------|----------|----------|-------|----------|
| i64         | i64    | str                                            | str    | f64   | i64   | i64   | str      | f64      | str   | str      |
| 892         | 3      | "Kelly, Mr. James"                           | "male" | 34.5  | 0     | 0     | "330911" | 7.8292   | null  | "Q"      |
| 893         | 3      | "Wilkes, Mrs. James (Ellen Needham)"        | "female" | 47.0  | 1     | 0     | "363272" | 7.0      | null  | "S"      |
| 894         | 2      | "Myles, Mr. Thomas Francis"                   | "male" | 62.0  | 0     | 0     | "240276" | 9.6875   | null  | "Q"      |
| 895         | 3      | "Wirz, Mr. Albert"                           | "male" | 27.0  | 0     | 0     | "315154" | 8.6625   | null  | "S"      |
| 896         | 3      | "Hirvonen, Mrs. Alexander (Helga)"           | "female" | 22.0  | 1     | 1     | "3101298"| 12.2875  | null  | "S"      |


| PassengerId | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket   | Fare     | Cabin | Embarked |
|-------------|--------|------------------------------------------------|--------|-------|-------|-------|----------|----------|-------|----------|
| i64         | i64    | str                                            | str    | f64   | i64   | i64   | str      | f64      | str   | str      |
| 892         | 3      | "Kelly, Mr. James"                           | "male" | 34.5  | 0     | 0     | "330911" | 7.8292   | null  | "Q"      |
| 893         | 3      | "Wilkes, Mrs. James (Ellen Needham)"        | "female" | 47.0  | 1     | 0     | "363272" | 7.0      | null  | "S"      |
| 894         | 2      | "Myles, Mr. Thomas Francis"                   | "male" | 62.0  | 0     | 0     | "240276" | 9.6875   | null  | "Q"      |
| 895         | 3      | "Wirz, Mr. Albert"                           | "male" | 27.0  | 0     | 0     | "315154" | 8.6625   | null  | "S"      |
| 896         | 3      | "Hirvonen, Mrs. Alexander (Helga)"           | "female" | 22.0  | 1     | 1     | "3101298"| 12.2875  | null  | "S"      |

Данные обучающей выборки:  
Размер: (891, 12)  

| PassengerId | Survived | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket              | Fare     | Cabin   | Embarked |
|-------------|----------|--------|------------------------------------------------|--------|-------|-------|-------|---------------------|----------|---------|----------|
| i64         | i64      | i64    | str                                            | str    | f64   | i64   | i64   | str                 | f64      | str     | str      |
| 1           | 0        | 3      | "Braund, Mr. Owen Harris"                     | "male" | 22.0  | 1     | 0     | "A/5 21171"         | 7.25     | "Unknown" | "S"      |
| 2           | 1        | 1      | "Cumings, Mrs. John Bradley (Flo)             | "female" | 38.0  | 1     | 0     | "PC 17599"          | 71.2833  | "C85"   | "C"      |
| 3           | 1        | 3      | "Heikkinen, Miss. Laina"                      | "female" | 26.0  | 0     | 0     | "STON/O2. 3101282"  | 7.925    | "Unknown" | "S"      |
| 4           | 1        | 1      | "Futrelle, Mrs. Jacques Heath (Flo)           | "female" | 35.0  | 1     | 0     | "113803"            | 53.1     | "C123"  | "S"      |
| 5           | 0        | 3      | "Allen, Mr. William Henry"                    | "male" | 35.0  | 0     | 0     | "373450"            | 8.05     | "Unknown" | "S"      |


Данные тестовой выборки:  
Размер: (418, 11)  

| PassengerId | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket   | Fare     | Cabin | Embarked |
|-------------|--------|------------------------------------------------|--------|-------|-------|-------|----------|----------|-------|----------|
| i64         | i64    | str                                            | str    | f64   | i64   | i64   | str      | f64      | str   | str      |
| 892         | 3      | "Kelly, Mr. James"                           | "male" | 34.5  | 0     | 0     | "330911" | 7.8292   | null  | "Q"      |
| 893         | 3      | "Wilkes, Mrs. James (Ellen Needham)"        | "female" | 47.0  | 1     | 0     | "363272" | 7.0      | null  | "S"      |
| 894         | 2      | "Myles, Mr. Thomas Francis"                   | "male" | 62.0  | 0     | 0     | "240276" | 9.6875   | null  | "Q"      |
| 895         | 3      | "Wirz, Mr. Albert"                           | "male" | 27.0  | 0     | 0     | "315154" | 8.6625   | null  | "S"      |
| 896         | 3      | "Hirvonen, Mrs. Alexander (Helga)"           | "female" | 22.0  | 1     | 1     | "3101298"| 12.2875  | null  | "S"      |

### Определение типов переменных


#### Бинарные переменные

In [23]:
# # Определение бинарных переменных и вывод их уникальных значений
# print("Бинарные переменные и их уникальные значения:")  # Заголовок для вывода

# # Перебор всех столбцов в данных
# for col in train_df.columns:
#     unique_values = train_df[col].unique().drop_nulls()  # Получение уникальных значений, игнорируя пропуски
#     if len(unique_values) == 2:  # Проверка, что в столбце ровно два уникальных значения
#         sorted_values = sorted(unique_values, key=lambda x: len(str(x)))  # Сортировка значений по длине строки
#         print(f"{col}: {sorted_values}")  # Вывод в формате "Название_колонки: [значение1, значение2]"

Бинарные переменные и их уникальные значения:  
Survived: [0, 1]  
Sex: ['male', 'female']

#### Непрерывные переменные

In [27]:
# Определение непрерывных переменных и вывод их уникальных значений
print("Непрерывные переменные и их уникальные значения:")  # Заголовок для вывода

# Перебор всех столбцов в данных
for col in train_df.columns:
    if train_df[col].dtype in [pl.Float64, pl.Int64]:  # Проверка, что столбец числовой (дробный или целый)
        unique_values = train_df[col].unique().drop_nulls()  # Получение уникальных значений, игнорируя пропуски
        if len(unique_values) > 2:  # Проверка, что в столбце больше двух уникальных значений (не бинарный)
            sorted_values = sorted(unique_values, key=lambda x: len(str(x)))  # Сортировка по длине строкового представления
            if len(sorted_values) > 10:  # Если значений больше 10, добавляем многоточие
                print(f"{col}: {sorted_values[:10]}...")
            else:  # Если значений 10 или меньше, выводим все
                print(f"{col}: {sorted_values}")

Непрерывные переменные и их уникальные значения:
PassengerId: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...
Pclass: [1, 2, 3]
Age: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.42]...
SibSp: [0, 1, 2, 3, 4, 5, 8]
Parch: [0, 1, 2, 3, 4, 5, 6]
Fare: [0.0, 5.0, 7.8, 8.3, 9.0, 9.5, 6.45, 6.75, 6.95, 7.05]...


```
Непрерывные переменные и их уникальные значения:  
PassengerId: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...  
Pclass: [1, 2, 3]  
Age: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.42]...  
SibSp: [0, 1, 2, 3, 4, 5, 8]  
Parch: [0, 1, 2, 3, 4, 5, 6]  
Fare: [0.0, 5.0, 7.8, 8.3, 9.0, 9.5, 6.45, 6.75, 6.95, 7.05]...
```

```
Непрерывные переменные и их уникальные значения:  
PassengerId: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...  
Pclass: [1, 2, 3]  
Age: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.42]...  
SibSp: [0, 1, 2, 3, 4, 5, 8]  
Parch: [0, 1, 2, 3, 4, 5, 6]  
Fare: [0.0, 5.0, 7.8, 8.3, 9.0, 9.5, 6.45, 6.75, 6.95, 7.05]...
```

`Непрерывные переменные и их уникальные значения:`  

`PassengerId: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...`  

`Pclass: [1, 2, 3]`  

`Age: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.42]...`  

`SibSp: [0, 1, 2, 3, 4, 5, 8]`  

`Parch: [0, 1, 2, 3, 4, 5, 6]`  

`Fare: [0.0, 5.0, 7.8, 8.3, 9.0, 9.5, 6.45, 6.75, 6.95, 7.05]...`